# Демо: потоки, процессы и GIL

Прокликай Shift+Enter каждую ячейку и сравни три способа выполнить работу: последовательно, через потоки, через процессы. На I/O-bound задаче потоки дают ускорение. На CPU-bound — нет (виноват GIL). А вот процессы ускоряют и то, и другое.

**Как читать этот ноутбук.** Часть примеров — обычные ячейки, запускаются прямо здесь по `Shift+Enter`. Часть (всё, что про `multiprocessing.Pool` и `multiprocessing.Process`) вынесена в `.py`-скрипты в папке `scripts/`. В таких местах ты увидишь блок:

> **→ Перейди в терминал.** `python scripts/<имя>.py`

Открой терминал в папке `notebooks/`, запусти указанный скрипт, сравни вывод с тем, что в ноутбуке, и возвращайся к следующей ячейке. Почему так — `multiprocessing.Pool/Process` в Jupyter на Windows не работает (дочерние процессы не видят функции из ячеек). Подробности — в `scripts/README.md`.

**Прочее:**

- Сравнение времени делаем приблизительно — точные числа зависят от загрузки машины и числа ядер.

## Часть 1. I/O-bound задача — последовательно

Симулируем сетевой запрос через `time.sleep(0.5)` — 500 мс ожидания ответа от сервера. Делаем 4 таких «запроса» подряд. Время — сумма.

In [1]:
import time

def fake_network_request(name):
    print(f"  start  {name}")
    time.sleep(0.5)             # имитация сетевой задержки
    print(f"  done   {name}")
    return f"<{name}>"

names = ["alpha", "beta", "gamma", "delta"]

start = time.perf_counter()
results = [fake_network_request(n) for n in names]
elapsed = time.perf_counter() - start
print(f"\nпоследовательно: {elapsed:.2f}s — сумма всех задержек")


  start  alpha


  done   alpha
  start  beta


  done   beta
  start  gamma


  done   gamma
  start  delta


  done   delta

последовательно: 2.01s — сумма всех задержек


## Часть 2. I/O-bound через потоки

Сейчас запустим те же 4 запроса в 4 отдельных потоках. Поток создаётся через `threading.Thread(target=func, args=(...))`, стартует через `start()`, и `join()` блокирует основной поток до его завершения.

Ожидание: общее время ≈ 0.5 с (все 4 ждут параллельно), а не 2.0 с.

In [2]:
import threading
import time

def fake_network_request(name):
    print(f"  start  {name}")
    time.sleep(0.5)
    print(f"  done   {name}")

names = ["alpha", "beta", "gamma", "delta"]

start = time.perf_counter()
threads = [threading.Thread(target=fake_network_request, args=(n,)) for n in names]
for t in threads:
    t.start()                    # все 4 потока стартуют почти одновременно
for t in threads:
    t.join()                     # ждём завершения всех
elapsed = time.perf_counter() - start
print(f"\nчерез потоки: {elapsed:.2f}s — близко к 0.5 (max задержка), не 2.0")


  start  alpha
  start  beta
  start  gamma
  start  delta


  done   beta
  done   alpha
  done   gamma
  done   delta

через потоки: 0.50s — близко к 0.5 (max задержка), не 2.0


Почему потоки помогли? `time.sleep` — блокирующий системный вызов: пока поток ждёт, GIL отпускается, и Python отдаёт управление другому потоку. Так все 4 потока одновременно «висят» в ожидании, и общее время — это максимальная задержка, а не сумма.

## Часть 3. CPU-bound через потоки — никакого ускорения

Теперь возьмём чистую CPU-задачу: посчитать `sum(i*i for i in range(N))`. Здесь нет блокирующих вызовов — просто арифметика на байткоде Python. GIL держит ровно один поток в активной фазе, остальные ждут.

Сравним 1 поток против 4 потоков на одной и той же общей работе.

In [3]:
import threading
import time

def cpu_task(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

N = 2_000_000

# Один поток на общую работу 4*N
start = time.perf_counter()
cpu_task(4 * N)
single_time = time.perf_counter() - start
print(f"1 поток ({4 * N} итераций): {single_time:.2f}s")

# 4 потока, каждый делает N — общий объём работы тот же
start = time.perf_counter()
threads = [threading.Thread(target=cpu_task, args=(N,)) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
multi_time = time.perf_counter() - start
print(f"4 потока × {N} итераций: {multi_time:.2f}s")

print(f"\nspeedup: {single_time / multi_time:.2f}x — близко к 1.0, потоки не помогают")


1 поток (8000000 итераций): 0.25s


4 потока × 2000000 итераций: 0.23s

speedup: 1.06x — близко к 1.0, потоки не помогают


Числа близки. На CPU-bound потоки **не дают параллелизма** — GIL запрещает байткоду выполняться одновременно из нескольких потоков. Они переключаются между собой, но в каждый момент работает один.

## Часть 4. CPU-bound через процессы — настоящий параллелизм

У `multiprocessing` другой механизм: каждый процесс — отдельный Python-интерпретатор со своим GIL. Они работают на разных ядрах CPU независимо, и общее время реально делится.

> **→ Перейди в терминал.** Этот пример не запускается из ноутбука — только из `.py`-скрипта. Открой терминал в папке `notebooks/` и выполни:
>
> ```bash
> python scripts/demo_pool_speedup.py
> ```
>
> Когда увидишь вывод — возвращайся сюда и читай дальше.

Ожидаемый вывод (точные числа зависят от машины):

```
Pool(4) × 2000000: 0.10s
1 процесс × 8000000: 0.27s

speedup: 2.69x — близко к числу ядер
```

На 4-ядерной машине ожидаем ускорения 2.5-4x (не идеально 4x — есть накладные на старт процессов и `pickle`-сериализацию данных между ними). Но это уже **настоящий параллелизм**, в отличие от потоков.

## Часть 5. Общая память: потоки ↔ процессы

Принципиальная разница: потоки **делят память**, процессы — **нет**. Покажем это: глобальная переменная, которую модифицирует поток, видна снаружи. Процесс модифицирует **копию**, оригинал не задет.

In [4]:
import threading

shared = {"count": 0}

def thread_worker():
    shared["count"] = 999

t = threading.Thread(target=thread_worker)
t.start()
t.join()

print("после потока shared:", shared)
# {'count': 999} — поток модифицировал ту же память


после потока shared: {'count': 999}


Вторую половину этого примера — `multiprocessing.Process` с изоляцией памяти — нужно посмотреть в `.py`-скрипте.

> **→ Перейди в терминал.** Из папки `notebooks/` запусти:
>
> ```bash
> python scripts/demo_process_memory.py
> ```
>
> Скрипт показывает контраст: поток меняет глобальный словарь родителя, процесс — только свою копию. Сравни вывод с ожидаемым ниже и возвращайся к шпаргалке.

Ожидаемый вывод:

```
после потока shared: {'count': 999}
  внутри процесса: {'count': 999}
после процесса shared: {'count': 0}
```

Эта разница определяет выбор:

- **Потоки** дешёвые в создании, шарят память — удобно, но опасно (race condition).
- **Процессы** дороже, изолированы — медленнее обмен данными (сериализация через `pickle`), зато безопасны и обходят GIL.

## Часть 6. Шпаргалка по выбору

| Тип задачи | Что брать | Почему |
|---|---|---|
| HTTP-запросы, запросы к БД, чтение/запись файлов | потоки или asyncio | I/O-bound — GIL отпускается на блокирующих вызовах |
| Перемножение матриц на чистом Python | процессы (`Pool`) | CPU-bound — потоки не параллелят |
| Числовая работа в NumPy / Pandas | потоки или процессы | NumPy сам отпускает GIL под ops, потоки помогают |
| Обработка картинок в Pillow | процессы | чисто CPU-bound, лучше параллелится |
| Веб-сервер с тысячами клиентов | asyncio | один event loop держит много соединений |
| Подготовка батчей в DataLoader | процессы (`num_workers=N`) | `pickle` для трансформов, GIL мешает с потоками |

Правило: задача упирается в I/O — потоки или asyncio. Задача упирается в CPU — процессы. Задача комбинированная — гибрид (`asyncio` + `asyncio.to_thread` для блокирующих кусков).

## Мини-задания

Три коротких упражнения. Подсказок к именам и API нет — вспомни сам.

**Задание 1.** Запусти 5 «сетевых запросов» (`time.sleep(0.3)` каждый) через `threading.Thread`. Замерь общее время — должно быть ~0.3с.

**Задание 2.** Возьми CPU-функцию (например, `sum(i*i for i in range(2_000_000))`). Сравни время прогона: 4 раза подряд через `for` (последовательно) против 4 раз параллельно через `multiprocessing.Pool(4).map`. Напечатай speedup.

> **→ Перейди в терминал.** Это задание решается в отдельном `.py`-файле. Открой `scripts/task_02_pool_practice.py` в редакторе, заполни блоки `TODO` и запусти из папки `notebooks/`:
>
> ```bash
> python scripts/task_02_pool_practice.py
> ```
>
> Когда заработает и увидишь speedup — сравни своё решение с `scripts/task_02_pool_solution.py`.

**Задание 3.** Что напечатает код ниже? Сначала угадай (помни про память потоков), потом запусти.

In [6]:
# Задание 1
import threading, time
def fake_request(name):
    time.sleep(0.3)
names = ['a', 'b', 'c', 'd', 'e']
# запусти 5 потоков и замерь время
start_time = time.perf_counter()
threads = [threading.Thread(target=fake_request, args=(n,)) for n in names]
for t in threads:
    t.start()
for t in threads:
    t.join()
print("Время выполнения:", time.perf_counter() - start_time)

Время выполнения: 0.305957708042115


In [ ]:
# Задание 3 — твой прогноз для каждой строки впиши в комментарий:
import threading

shared_list = []

def append_one(value):
    shared_list.append(value)

threads = [threading.Thread(target=append_one, args=(i,)) for i in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(shared_list)           # ?
print(len(shared_list))      # ?
print(sorted(shared_list))   # ?  -- порядок исходно случайный


[0, 1, 2, 3, 4]
5
[0, 1, 2, 3, 4]
